# credit-risk-scorecard: 파이프라인 워크스루

`src/` 모듈을 직접 불러와 전처리 -> WoE/IV -> 베이스라인 스코어카드 -> LightGBM -> 챔피언-챌린저 -> 등급/컷오프 시뮬레이션 흐름을 단계별로 확인합니다.

전체 자동 실행은 저장소 루트의 `run.py`를 참고하세요.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src import preprocess, scoring, simulation, train_advanced, train_baseline
from src.db import get_connection, init_db

pd.set_option('display.max_columns', 50)

## 1. 데이터 로드 & 전처리

In [ ]:
raw_df = preprocess.load_raw()
df = preprocess.basic_preprocess(raw_df)
print(raw_df.shape, '->', df.shape)
df.head()

## 2. SQLite 저장 (applicants)

In [ ]:
init_db()
conn = get_connection()
preprocess.save_applicants(df, conn)
pd.read_sql('SELECT COUNT(*) AS n FROM applicants', conn)

## 3. WoE/IV 계산 + 로지스틱 베이스라인 스코어카드

In [ ]:
baseline = train_baseline.run_baseline(df, conn)
print('AUC', baseline['auc'], 'KS', baseline['ks'], 'Gini', baseline['gini'])
baseline['iv_summary'].head(15)

In [ ]:
pd.read_sql('SELECT * FROM iv_summary ORDER BY iv_value DESC LIMIT 15', conn)

## 4. LightGBM 고도화 모델

In [ ]:
advanced = train_advanced.run_lightgbm(baseline['train_df'], baseline['test_df'], conn)
print('AUC', advanced['auc'], 'KS', advanced['ks'], 'Gini', advanced['gini'])

## 5. 챔피언-챌린저 비교

In [ ]:
champion = train_advanced.choose_champion(baseline, advanced)
champion

## 6. 점수 스케일링(PDO) 및 A-E 등급

In [ ]:
if champion['champion'] == 'lightgbm':
    champion_run_id = advanced['run_id']
    full_proba = train_advanced.score_full_population(advanced['model'], df, advanced['feature_cols'], advanced['cat_cols']).to_numpy()
    ids_full = df[train_baseline.ID_COL].to_numpy()
    target_full = df[train_baseline.TARGET_COL].to_numpy()
else:
    champion_run_id = baseline['run_id']
    full_proba = baseline['proba_full']
    ids_full = baseline['ids_full']
    target_full = baseline['target_full']

full_score = scoring.probability_to_score(full_proba)
score_series = pd.Series(full_score)
grades = scoring.assign_grades(score_series)
scoring.save_scores(ids_full, full_score, grades, champion_run_id, conn)

grade_df = pd.DataFrame({'grade': grades.values, 'score': full_score, 'target': target_full})
grade_stats = grade_df.groupby('grade').agg(count=('target', 'size'), avg_score=('score', 'mean'), default_rate=('target', 'mean')).reindex(scoring.GRADE_LABELS_ASCENDING[::-1])
grade_stats

## 7. 컷오프 시뮬레이션

In [ ]:
sim_df = simulation.simulate_cutoffs(score_series, pd.Series(target_full))
simulation.save_cutoff_simulation(sim_df, champion_run_id, conn)
sim_df

In [ ]:
conn.close()